In [1]:
import pandas as pd
import huggingface_hub
from huggingface_hub import login
from datasets import load_dataset
from transformers import AutoTokenizer
import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForMaskedLM, DataCollatorForLanguageModeling
from sklearn.model_selection import train_test_split
import os
import numpy as np

os.environ["http_proxy"] = "http://127.0.0.1:10809"
os.environ["https_proxy"] = "http://127.0.0.1:10809" 


c:\Users\DELL\anaconda3\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.2
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
# Upload dataset to huggingface
!huggingface-cli lfs-enable-largefiles  # needed if some files are bigger than 5Gb

usage: huggingface-cli <command> [<args>]
huggingface-cli: error: unrecognized arguments: needed if some files are bigger than 5Gb


In [2]:
login(token = "HF_TOKEN_REMOVED", add_to_git_credential=True)

Token is valid.
Your token has been saved in your configured git credential helpers (manager).
Your token has been saved to C:\Users\DELL\.cache\huggingface\token
Login successful


### The repo is already created and do not need to be created again

In [3]:
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(repo_id="job-posting-soc")

RepoUrl('https://huggingface.co/Zexuan/job-posting-soc', endpoint='https://huggingface.co', repo_type='model', repo_id='Zexuan/job-posting-soc')

### Upload the large file to the repo using Git LFS

In [10]:
from huggingface_hub import HfApi
api = HfApi()

api.upload_file(
    path_or_fileobj="F:/Data/job_posting/processed/finetune/df_titleLabel.csv",
    path_in_repo="df_titleLabel.csv",
    repo_id="Zexuan/soc_data",
    repo_type="dataset",
)

df_titleLabel.csv: 100%|██████████| 31.7G/31.7G [1:39:34<00:00, 5.30MB/s]
Upload 1 LFS files: 100%|██████████| 1/1 [1:39:34<00:00, 5974.96s/it]


'https://huggingface.co/datasets/Zexuan/soc_data/blob/main/df_titleLabel.csv'

In [15]:
df = pd.read_csv("F:/Data/job_posting/processed/finetune/test_df_sample.csv", nrows = 5)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  2 non-null      float64
 1   招聘主键ID      5 non-null      int64  
 2   工作描述        5 non-null      object 
 3   工作名称        5 non-null      object 
 4   soc_code    5 non-null      int64  
 5   true_ind    5 non-null      bool   
dtypes: bool(1), float64(1), int64(2), object(2)
memory usage: 333.0+ bytes


### Finetune a model

In [14]:
# get the ONET SOC Code (the real one)
df_soc = pd.read_csv('F:/Data/job_posting/processed/2019_to_SOC_Crosswalk.csv')
# keep column '2018 SOC Code'
df_soc = df_soc[['2018 SOC Code']]
# replace the last digit of 'soc_code' with '0'
df_soc['2018 SOC Code'] = df_soc['2018 SOC Code'].str[:-1] + '0'
# rename the column name to 'soc_code'
df_soc.rename(columns={'2018 SOC Code': 'soc_code'}, inplace=True)
df_soc = df_soc.drop_duplicates(subset=['soc_code'], keep='first')
len(df_soc['soc_code'].unique())

FileNotFoundError: [Errno 2] No such file or directory: 'F:/Data/job_posting/processed/2019_to_SOC_Crosswalk.csv'

### load the related data when needed

In [14]:
import csv
df_titlelabel= pd.read_csv('F:/Data/job_posting/processed/finetune/df_titleLabel.csv', encoding = "utf_8_sig", on_bad_lines='skip', encoding_errors='ignore')
#train_df = pd.read_csv('F:/Data/job_posting/processed/finetune/train_df.csv', encoding = "utf_8_sig", on_bad_lines='skip', encoding_errors='ignore')
#test_df = pd.read_csv('F:/Data/job_posting/processed/finetune/test_df.csv', encoding = "utf_8_sig", on_bad_lines='skip', encoding_errors='ignore')
#train_df_sample = pd.read_csv('E:/Data/job_posting/processed/finetune/train_df_sample.csv', encoding = "utf_8_sig", on_bad_lines='skip', encoding_errors='ignore')
#test_df_sample = pd.read_csv('E:/Data/job_posting/processed/finetune/test_df_sample.csv', encoding = "utf_8_sig", on_bad_lines='skip', encoding_errors='ignore')
train_df, test_df = train_test_split(df_titlelabel, test_size=0.3, random_state=42)
train_df.to_csv('F:/Data/job_posting/processed/finetune/train_df.csv', index=False, encoding = "utf_8_sig", header=True, quoting=csv.QUOTE_NONNUMERIC)
test_df.to_csv('F:/Data/job_posting/processed/finetune/test_df.csv', index=False, encoding = "utf_8_sig", header=True, quoting=csv.QUOTE_NONNUMERIC)

,招聘主键ID,工作描述,工作名称,soc_code
0,133575,技能要求：招聘（管培生招聘方向），员工关系岗位职责：1、负责集团管理培训生、储备干部的人才引...,招聘专员/高级招聘专员,13-1070
1,142723,多渠道为公司寻求合适的人才。,招聘专员/高级招聘专员,13-1070
2,548307,招聘专员岗位:1.负责公司内部员工销售，外贸业务员，电商运营等岗位招聘2.负责合作单位及客户...,招聘专员/高级招聘专员,13-1070
3,592889,岗位职责：1、负责集团管理培训生、储备干部等岗位的人才引进、配置工作；2、通过多种渠道，高效...,招聘专员/高级招聘专员,13-1070
4,645977,岗位职责：1、维护和拓展招聘渠道，每天更新、发布招聘网站职位信息；2、筛选简历，电话邀约、组...,招聘专员/高级招聘专员,13-1070
...,...,...,...,...
37889663,156486445,职位要求：48岁以下，身高168以上，身体健康，户籍不限；无不良嗜好；人品好1.太平间值班员...,太平间殡仪馆陵园代客祭祀,21-2020
37889664,156486447,职位要求：48岁以下，身高168以上，身体健康，户籍不限；无不良嗜好；人品好1.太平间值班员...,太平间殡仪馆陵园代客祭祀,21-2020
37889665,156486448,职位要求：48岁以下，身高168以上，身体健康，户籍不限；无不良嗜好；人品好1.太平间值班员...,太平间殡仪馆陵园代客祭祀,21-2020
37889666,156486449,职位要求：48岁以下，身高168以上，身体健康，户籍不限；无不良嗜好；人品好1.太平间值班员...,太平间殡仪馆陵园代客祭祀,21-2020


In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW, AutoModelForMaskedLM
from transformers import BertForSequenceClassification
import torch
from transformers import AdamW
from transformers import get_scheduler
from transformers import Trainer
from transformers import BertTokenizer
import numpy as np
import pandas as pd


# model = AutoModelForSequenceClassification.from_pretrained("bert-base-chinese")
# model = BertForSequenceClassification.from_pretrained("hfl/chinese-bert-wwm")

#tokenizer = AutoTokenizer.from_pretrained("bert-base-chinese")
tokenizer = BertTokenizer.from_pretrained("bert-base-chinese")

In [ ]:
train_df_sample = pd.read_csv('D:/Data/job_posting/processed/finetune/train_df.csv', encoding = "utf_8_sig", on_bad_lines='skip', encoding_errors='ignore')
# replace the last digit of 'soc_code' with '0'
train_df_sample['soc_code'] = train_df_sample['soc_code'].str[:-1] + '0'
# merge the 'test_dfSoc' and 'df_soc' using 'soc_code', only keep the matched sample
train_df_sample = pd.merge(train_df_sample, df_soc, on='soc_code', how='inner')

In [ ]:
test_df_sample = pd.read_csv('F:/Data/job_posting/processed/finetune/test_df.csv', encoding = "utf_8_sig", on_bad_lines='skip', encoding_errors='ignore')
# replace the last digit of 'soc_code' with '0'
test_df_sample['soc_code'] = test_df_sample['soc_code'].str[:-1] + '0'
# merge the 'test_dfSoc' and 'df_soc' using 'soc_code', only keep the matched sample
test_df_sample = pd.merge(test_df_sample, df_soc, on='soc_code', how='inner')

In [6]:
# keep the overalp soc_code in train and test dataset
train_df_sample = train_df_sample[train_df_sample['soc_code'].isin(test_df_sample['soc_code'])]
test_df_sample = test_df_sample[test_df_sample['soc_code'].isin(train_df_sample['soc_code'])]

# replace the symbol '-' to '.' in soc_code column, and convert soc_code to int
train_df_sample['soc_code'] = train_df_sample['soc_code'].str.replace('-', '').astype(int)
test_df_sample['soc_code'] = test_df_sample['soc_code'].str.replace('-', '').astype(int)

# generate a new column 'soc_code1' with value to recode the 'soc_code' in ascending order
train_df_sample['soc_code1'] = train_df_sample['soc_code'].rank(method='dense').astype(int)
test_df_sample['soc_code1'] = test_df_sample['soc_code'].rank(method='dense').astype(int)

# recode the 'soc_code 1' so that it starts from value 0
train_df_sample['soc_code1'] = train_df_sample['soc_code1'] - train_df_sample['soc_code1'].min()
test_df_sample['soc_code1'] = test_df_sample['soc_code1'] - test_df_sample['soc_code1'].min()

# convert '工作描述' to string
train_df_sample['工作描述'] = train_df_sample['工作描述'].astype(str)
test_df_sample['工作描述'] = test_df_sample['工作描述'].astype(str)


In [7]:

# Tokenize the text and convert it into input features
train_texts = train_df_sample['工作描述'].tolist()
train_labels = train_df_sample['soc_code1'].tolist()
train_encodings = tokenizer(train_texts, truncation=True, padding=True)

test_texts = test_df_sample['工作描述'].tolist()
test_labels = test_df_sample['soc_code1'].tolist()
test_encodings = tokenizer(test_texts, truncation=True, padding=True)

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import BertForSequenceClassification, AdamW, get_linear_schedule_with_warmup

# Convert the input features into PyTorch tensors
train_inputs = torch.tensor(train_encodings['input_ids'])
train_masks = torch.tensor(train_encodings['attention_mask'])
train_labels = torch.tensor(train_labels)

test_inputs = torch.tensor(test_encodings['input_ids'])
test_masks = torch.tensor(test_encodings['attention_mask'])
test_labels = torch.tensor(test_labels)


In [ ]:
# Create a PyTorch DataLoader to iterate over the data during training
batch_size = 32

train_data = TensorDataset(train_inputs, train_masks, train_labels)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)

test_data = TensorDataset(test_inputs, test_masks, test_labels)
test_loader = DataLoader(test_data, batch_size=batch_size)

# Define the model, the optimizer and the learning rate scheduler
num_labels = len(train_df_sample['soc_code1'].unique())
model = BertForSequenceClassification.from_pretrained("hfl/chinese-bert-wwm", num_labels=num_labels)
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)
num_epochs = 4
num_training_steps = num_epochs * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

# Train the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.train()
for epoch in range(num_epochs):
    for batch in train_loader:
        inputs = batch[0].to(device)
        masks = batch[1].to(device)
        labels = batch[2].to(device)

        outputs = model(inputs, attention_mask=masks, labels=labels)
        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()


In [ ]:
# Evaluate the performance of the model on the test set
model.eval()
predictions = []
with torch.no_grad():
    for batch in test_loader:
        inputs = batch[0].to(device)
        masks = batch[1].to(device)

        # Resize the attention mask tensor to match the size of the inputs
        # masks.resize_(inputs.shape[0], inputs.shape[1])
    
        outputs = model(inputs, attention_mask=masks)
        logits = outputs.logits
        batch_predictions = torch.argmax(logits, axis=1).cpu().numpy()
        predictions.extend(batch_predictions)

from sklearn.metrics import accuracy_score
accuracy = accuracy_score(test_labels, predictions)
print(f"Accuracy: {accuracy}")